# OOP 5: Encapsulation, Abstraction, and Layered Architecture

## Table of Contents

1. [Encapsulation: Controlling Access](#1)
2. [Why It Matters — The Data Leakage Example](#2)
3. [Properties: Read-Only State](#3)
4. [Abstraction: Enforcing Interfaces with ABCs](#4)
5. [Layered Architecture](#5)
6. [Encapsulation Is What Makes Layers Real](#6)
7. [Exercise: Harden the Feature Store](#7)

---
## 1. Encapsulation: Controlling Access <a id="1"></a>

In notebook 02 you built subclasses with `_fitted`, `_mean`, `_std`. That underscore wasn't accidental — it's **encapsulation**: the decision about what the outside world is allowed to see.

Python uses naming conventions:

| Convention | Meaning | Rule |
|---|---|---|
| `name` | Public | Part of the API — safe to depend on |
| `_name` | Protected | Internal detail — don't depend on it from outside |
| `__name` | Private (name-mangled) | Python renames it to `_ClassName__name` |

The single underscore `_` covers 95% of cases. It's a social contract: *"I reserve the right to change this without notice."*

In [ ]:
import pandas as pd
import numpy as np
from abc import ABC, abstractmethod
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

class StandardScaler:
    def __init__(self):
        self._mean = None    # protected: learned during fit
        self._std = None     # protected: learned during fit
        self._fitted = False # protected: internal state flag

    def fit(self, X: pd.DataFrame) -> "StandardScaler":
        self._mean = X.mean()
        self._std = X.std().clip(lower=1e-8)
        self._fitted = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not self._fitted:
            raise RuntimeError("Call fit() before transform().")
        return (X - self._mean) / self._std

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return self.fit(X).transform(X)


X_train = pd.DataFrame({"tenure": [12, 5, 24, 3, 8], "charges": [70, 30, 90, 20, 55]})
X_test  = pd.DataFrame({"tenure": [17, 1],            "charges": [80, 15]})

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # uses stats from training only — no leakage

print(X_train_scaled.round(2))

---
## 2. Why It Matters — The Data Leakage Example <a id="2"></a>

Here is the same scaler without encapsulation. Public attributes invite a silent, damaging bug.

In [ ]:
class LeakyScaler:
    def __init__(self):
        self.mean = None   # public — anyone can overwrite
        self.std = None

    def fit(self, X):
        self.mean = X.mean()
        self.std = X.std()
        return self

    def transform(self, X):
        return (X - self.mean) / self.std


np.random.seed(0)
X_tr = pd.DataFrame({"tenure": np.random.randint(1, 60, 100)})
X_te = pd.DataFrame({"tenure": np.random.randint(1, 60, 20)})

bad_scaler = LeakyScaler()
bad_scaler.fit(X_tr)

# A well-meaning colleague "updates" the mean for the test set...
bad_scaler.mean = X_te.mean()   # silent corruption — no error, no warning

result = bad_scaler.transform(X_te)   # uses test statistics: data leakage

print("Mean used:", bad_scaler.mean.values)
print("Should be:", X_tr.mean().values)
print()
print("The model trained on this reports optimistic evaluation metrics,")
print("then fails in production where test statistics are unavailable.")

---
## 3. Properties: Read-Only State <a id="3"></a>

Sometimes you *want* callers to read an internal value — just not write it.
`@property` gives you a read-only attribute that's computed on access.

In [ ]:
class StandardScaler:
    def __init__(self):
        self._mean = None
        self._std = None
        self._fitted = False

    def fit(self, X: pd.DataFrame) -> "StandardScaler":
        self._mean = X.mean()
        self._std = X.std().clip(lower=1e-8)
        self._fitted = True
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if not self._fitted:
            raise RuntimeError("Call fit() before transform().")
        return (X - self._mean) / self._std

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return self.fit(X).transform(X)

    @property
    def is_fitted(self) -> bool:
        return self._fitted

    @property
    def feature_names(self) -> list[str] | None:
        return list(self._mean.index) if self._fitted else None


X = pd.DataFrame({"tenure": [12, 5, 24, 3, 8], "charges": [70, 30, 90, 20, 55]})
scaler = StandardScaler()

print("Before fit:", scaler.is_fitted, scaler.feature_names)
scaler.fit(X)
print("After fit: ", scaler.is_fitted, scaler.feature_names)

# Properties are read-only — writing raises AttributeError
try:
    scaler.is_fitted = False
except AttributeError as e:
    print(f"AttributeError: {e}")

---
## 4. Abstraction: Enforcing Interfaces with ABCs <a id="4"></a>

In notebook 02 we used `raise NotImplementedError` to signal that subclasses should implement `fit()` and `transform()`. This works, but the failure happens **at call time** — deep in a pipeline, with a confusing error.

**Abstract Base Classes (ABCs)** fail earlier: at **instantiation time**. Python refuses to create an object from any subclass that hasn't implemented the required methods.

This is **abstraction**: you declare *what* a class must do, without specifying *how*.

In [ ]:
class BasePreprocessor(ABC):
    """Interface contract for all preprocessing steps.

    Any subclass must implement fit() and transform().
    fit_transform() is provided as a template method.
    """

    @abstractmethod
    def fit(self, X: pd.DataFrame) -> "BasePreprocessor":
        """Learn parameters from X. Return self."""
        ...

    @abstractmethod
    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Apply learned transformation to X."""
        ...

    def fit_transform(self, X: pd.DataFrame) -> pd.DataFrame:
        return self.fit(X).transform(X)

    def __repr__(self) -> str:
        return f"{type(self).__name__}()"


# Instantiating the ABC itself fails immediately
try:
    p = BasePreprocessor()
except TypeError as e:
    print(f"TypeError: {e}")

# A subclass that forgets transform() also fails at instantiation
class IncompleteScaler(BasePreprocessor):
    def fit(self, X): return self
    # transform() missing

try:
    s = IncompleteScaler()
except TypeError as e:
    print(f"TypeError: {e}")

print()
print("Caught at object creation — not somewhere downstream at 2am.")

In [ ]:
# A complete subclass — must implement both methods
class StandardScaler(BasePreprocessor):
    def __init__(self):
        self._mean = None
        self._std = None

    def fit(self, X: pd.DataFrame) -> "StandardScaler":
        self._mean = X.mean()
        self._std = X.std().clip(lower=1e-8)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if self._mean is None: raise RuntimeError("fit() first.")
        return (X - self._mean) / self._std


class MinMaxScaler(BasePreprocessor):
    def __init__(self):
        self._min = None
        self._range = None

    def fit(self, X: pd.DataFrame) -> "MinMaxScaler":
        self._min = X.min()
        self._range = (X.max() - X.min()).clip(lower=1e-8)
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if self._min is None: raise RuntimeError("fit() first.")
        return (X - self._min) / self._range


# Both share the same interface
X = pd.DataFrame({"tenure": [12, 5, 24, 3, 8], "charges": [70, 30, 90, 20, 55]})
for Cls in [StandardScaler, MinMaxScaler]:
    print(f"{Cls.__name__}:")
    print(Cls().fit_transform(X).round(2))
    print()

---
## 5. Layered Architecture <a id="5"></a>

A layered architecture divides code into tiers where **each layer only talks to the one directly below it**.

For a DS pipeline:

```
┌─────────────────────────────────────┐
│         User / API Layer            │  run_experiment(), CLI, REST endpoint
├─────────────────────────────────────┤
│       Business Logic Layer          │  ChurnModel, training loop, evaluation
├─────────────────────────────────────┤
│          Data Layer                 │  DataLoader, FeatureStore
└─────────────────────────────────────┘
```

**Why this matters:**
- Swap CSV loading for a database? Change only the data layer
- Add a REST API? Add only a new user layer
- Test the business logic? Mock the data layer — no real files needed

This is how web backends have been built for 30 years. ML pipelines benefit from exactly the same discipline.

In [ ]:
# ── Data Layer ───────────────────────────────────────────────────────────
# Knows about: files, databases, data shapes.
# Does not know about: model training, evaluation, experiments.

class BaseLoader(ABC):
    @abstractmethod
    def load(self) -> pd.DataFrame: ...

    def row_count(self, df: pd.DataFrame) -> int:
        return len(df)


class CSVLoader(BaseLoader):
    def __init__(self, path: str):
        self._path = path   # protected: implementation detail

    def load(self) -> pd.DataFrame:
        # Simulate loading — in real code: pd.read_csv(self._path)
        np.random.seed(42)
        n = 200
        df = pd.DataFrame({
            "customerID":     [f"C{i:04d}" for i in range(n)],
            "tenure":         np.random.randint(1, 72, n).astype(float),
            "MonthlyCharges": np.random.uniform(20, 100, n),
            "TotalCharges":   np.random.uniform(100, 5000, n).astype(str),
            "Churn":          np.random.choice(["Yes", "No"], n),
        })
        df.loc[df.sample(10, random_state=1).index, "tenure"] = np.nan
        return df


class FeatureStore:
    REQUIRED_COLS = ["tenure", "MonthlyCharges", "TotalCharges", "Churn"]

    def __init__(self):
        self._df: pd.DataFrame | None = None   # protected

    def prepare(self, raw: pd.DataFrame) -> pd.DataFrame:
        missing = [c for c in self.REQUIRED_COLS if c not in raw.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")
        df = raw.drop(columns=["customerID"], errors="ignore").copy()
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
        self._df = df.dropna()
        return self._df

    @property
    def shape(self) -> tuple | None:
        return self._df.shape if self._df is not None else None

In [ ]:
# ── Business Logic Layer ─────────────────────────────────────────────────
# Knows about: ML, training, evaluation.
# Does not know about: where data comes from or how it was cleaned.

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report

class ChurnModel:
    FEATURE_COLS = ["tenure", "MonthlyCharges", "TotalCharges"]
    TARGET_COL = "Churn"

    def __init__(self, n_estimators: int = 100, random_state: int = 42):
        self.n_estimators = n_estimators
        self.random_state = random_state
        self._model: RandomForestClassifier | None = None  # protected
        self._encoder = LabelEncoder()                     # protected

    def train(self, df: pd.DataFrame) -> "ChurnModel":
        X = df[self.FEATURE_COLS]
        y = self._encoder.fit_transform(df[self.TARGET_COL])
        X_tr, self._X_te, y_tr, self._y_te = train_test_split(
            X, y, test_size=0.2, random_state=self.random_state
        )
        self._model = RandomForestClassifier(
            n_estimators=self.n_estimators, random_state=self.random_state
        )
        self._model.fit(X_tr, y_tr)
        return self

    def evaluate(self) -> None:
        preds = self._model.predict(self._X_te)
        print(classification_report(self._y_te, preds))

    @property
    def is_trained(self) -> bool:
        return self._model is not None

In [ ]:
# ── User / API Layer ─────────────────────────────────────────────────────
# Knows about: the full picture — wires all layers together.
# This is the only class that imports from both layers above.

class ExperimentRunner:
    """Orchestrates a full training run.

    Wires DataLoader → FeatureStore → ChurnModel.
    None of the layers below need to know about each other.
    """

    def __init__(self, source: str, n_estimators: int = 100):
        self._loader = CSVLoader(source)
        self._store  = FeatureStore()
        self._model  = ChurnModel(n_estimators=n_estimators)

    def run(self) -> None:
        raw   = self._loader.load()
        clean = self._store.prepare(raw)
        self._model.train(clean)

        print(f"Loaded {self._loader.row_count(raw)} rows")
        print(f"After cleaning: {self._store.shape}")
        print()
        self._model.evaluate()


runner = ExperimentRunner(source="data/raw/churn.csv", n_estimators=50)
runner.run()

---
## 6. Encapsulation Is What Makes Layers Real <a id="6"></a>

Look at what just happened in the code above:

- `ExperimentRunner` cannot accidentally reach into `CSVLoader._path` — it's protected
- `ChurnModel` cannot accidentally overwrite `FeatureStore._df` — they never share a reference
- Each layer exposes exactly one public method: `load()`, `prepare()`, `train()`

**Without encapsulation, layers are just comments.** Code can be organised into three files labelled "data layer", "business layer", "user layer" — and still be completely tangled if attributes are public and any layer can poke at any other's state.

With encapsulation, the architecture is enforced by the code itself:

```
ExperimentRunner          ← sees: load(), prepare(), train(), evaluate()
    │
    ├── CSVLoader         ← hides: _path
    ├── FeatureStore      ← hides: _df
    └── ChurnModel        ← hides: _model, _encoder, _X_te, _y_te
```

ABCs add one more guarantee: the data layer can be swapped (CSV → database → S3) and the business logic layer will never notice, because both implementations honour the `BaseLoader` contract.

---
## 7. Exercise: Harden the Feature Store <a id="7"></a>

The `FeatureStore` above validates columns but is still missing a few things.

1. Add a `@property columns` that returns the column list after preparation, or `None` if not yet prepared
2. Add a `__repr__` that shows readiness and shape: `FeatureStore(not prepared)` or `FeatureStore(shape=(180, 4))`
3. Add a `DatabaseLoader(BaseLoader)` stub — it should accept a `connection_string` and `table` name, and `load()` should raise `NotImplementedError` with a message like *"DatabaseLoader.load() not yet implemented — use CSVLoader for now"*

Then confirm: does `ExperimentRunner` need any changes to work with `DatabaseLoader` once it's fully implemented? Why or why not?

In [ ]:
# 1 + 2: extend FeatureStore
class FeatureStore:
    REQUIRED_COLS = ["tenure", "MonthlyCharges", "TotalCharges", "Churn"]

    def __init__(self):
        self._df: pd.DataFrame | None = None

    def prepare(self, raw: pd.DataFrame) -> pd.DataFrame:
        missing = [c for c in self.REQUIRED_COLS if c not in raw.columns]
        if missing:
            raise ValueError(f"Missing required columns: {missing}")
        df = raw.drop(columns=["customerID"], errors="ignore").copy()
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
        self._df = df.dropna()
        return self._df

    @property
    def columns(self) -> list[str] | None:
        pass  # TODO

    def __repr__(self) -> str:
        pass  # TODO


# 3: DatabaseLoader stub
class DatabaseLoader(BaseLoader):
    def __init__(self, connection_string: str, table: str):
        pass  # TODO

    def load(self) -> pd.DataFrame:
        pass  # TODO — raise NotImplementedError


# --- Tests ---
loader = CSVLoader("demo")
raw = loader.load()

fs = FeatureStore()
print(repr(fs))
fs.prepare(raw)
print(repr(fs))
print("Columns:", fs.columns)

try:
    db = DatabaseLoader("postgresql://localhost/churn", "customers")
    db.load()
except NotImplementedError as e:
    print(f"NotImplementedError: {e}")

> <details>
> <summary>Solution</summary>
>
> ```python
>     @property
>     def columns(self) -> list[str] | None:
>         return list(self._df.columns) if self._df is not None else None
>
>     def __repr__(self) -> str:
>         if self._df is None:
>             return "FeatureStore(not prepared)"
>         return f"FeatureStore(shape={self._df.shape})"
>
>
> class DatabaseLoader(BaseLoader):
>     def __init__(self, connection_string: str, table: str):
>         self._connection_string = connection_string
>         self._table = table
>
>     def load(self) -> pd.DataFrame:
>         raise NotImplementedError(
>             "DatabaseLoader.load() not yet implemented — use CSVLoader for now"
>         )
> ```
>
> **ExperimentRunner needs zero changes.** It depends on `BaseLoader`, not on `CSVLoader`. Once `DatabaseLoader.load()` is implemented, you pass it in and everything works. That's the payoff of the ABC contract.
> </details>